# v30 — реранкинг только внутри raw top-10/20/50

**Restart Kernel → Run All** в прежнем research `.venv`. Сравниваем четыре фиксированных варианта: текущий v25 и ограничение его графового порядка исходным top-10, top-20, top-50.

Гипотеза: ограничение пула сохранит часть правильных находок, которые полный реранкинг вытесняет. Это не гарантированный прирост. Используются готовые признаки: нового обучения и извлечения признаков нет.

Выбор — по calibration. На validation проверяем только v25 и один заранее зафиксированный выбор. Данные уже наблюдались в исследовании: это не независимая оценка hidden test. Рабочий MVP автоматически не меняется.

In [1]:
from pathlib import Path
import os, sys, subprocess
# До любого импорта ONNX Runtime: предотвращает известный native SIGABRT на macOS.
if 'onnxruntime' in sys.modules and os.environ.get('ORT_DISABLE_TELEMETRY') != '1':
    raise RuntimeError('ONNX Runtime уже загружен. Выполни Restart Kernel → Run All')
os.environ['ORT_DISABLE_TELEMETRY'] = '1'
candidates = [Path.cwd(), Path.cwd() / 'Car-classification-MSK', *Path.cwd().parents]
REPO = next((p for p in candidates if (p / 'training/pool_rerank.py').is_file()), None)
if REPO is None:
    raise RuntimeError('Открой notebook из исследовательского Car-classification-MSK')
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from IPython.display import Markdown, display
from training import pool_rerank as experiment
RUN_NAME = 'pool_rerank_v1'
SOURCE_RUN = 'family_finalists_v1'
ALLOW_OUTER_EVALUATION = True
print('Kernel:', sys.executable, '\nRun:', RUN_NAME)
display(experiment.systems())
print('CPU; готовые эмбеддинги; без обучения и лимита времени; готовые этапы возобновляются')

Kernel: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/.venv/bin/python 
Run: pool_rerank_v1


[{'name': 'V25_control', 'pool_k': None},
 {'name': 'raw_pool10', 'pool_k': 10},
 {'name': 'raw_pool20', 'pool_k': 20},
 {'name': 'raw_pool50', 'pool_k': 50}]

CPU; готовые эмбеддинги; без обучения и лимита времени; готовые этапы возобновляются


## 1. Проверки компонентов

Проверяем состав пула и неизменность его хвоста, порядок v25, отдельный raw R1-кандидат, независимость query, экспорт, запрет дополнительных validation-вариантов, продолжение после сбоя. Ошибка тестов останавливает запуск.

In [2]:
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_pool_rerank.py', 'tests/test_map_search.py', 'tests/test_dual_role_inference.py'], check=True, env={**os.environ, 'ORT_DISABLE_TELEMETRY': '1'})

..................................................                       [100%]
50 passed in 13.37s


CompletedProcess(args=['/Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/.venv/bin/python', '-m', 'pytest', '-q', 'tests/test_pool_rerank.py', 'tests/test_map_search.py', 'tests/test_dual_role_inference.py'], returncode=0)

## 2. Preflight и фиксация четырёх сравнений

Проверяем завершённый v29, контроль v25, контрольные суммы кэша/весов/исходников и исходные протоколы. Фиксируем новый каталог v30, ничего не перезаписывая в прошлых запусках. Проверка больших файлов может занять некоторое время.

In [3]:
context = experiment.prepare(RUN_NAME, source_run=SOURCE_RUN)
print('Результаты:', context['output'])
print('Signature:', context['signature'])
display(context['manifest']['pool_rerank_plan'])

Preflight OK: v25 + raw top10/20/50; cached embeddings; no training or automatic promotion
Результаты: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_30_pool_rerank/runs/pool_rerank_v1
Signature: 467976b0985cbace3d9915df2b578c755e1abd48bc9cf19815e84e9987813a35


{'source_directory': '/Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_29_family_finalists/runs/family_finalists_v1',
 'source_signature': 'cd81a4f6df0215c5c28b57803f38d7a4ef1076525ead2d046c77fc69825a35b8',
 'systems': [{'name': 'V25_control', 'pool_k': None},
  {'name': 'raw_pool10', 'pool_k': 10},
  {'name': 'raw_pool20', 'pool_k': 20},
  {'name': 'raw_pool50', 'pool_k': 50}],
 'pool': 'top K of original normalized MVP/R1 50:50 cosine fusion; K=10/20/50',
 'reranking': 'restrict full static-gallery v25 k20/k3/lambda.50 order to the raw pool',
 'tail': 'unchanged raw-fusion order outside the pool; no outsiders enter the pool',
 'candidate': 'original raw R1 candidate/cosine/refusal and frozen threshold',
 'selection': 'maximum calibration mAP over all four systems; ties retain v25 first',
 'validation': 'v25 and one frozen calibration winner only; no automatic promotion',
 'scope': 'post-hoc search on observed development

## 3. Calibration → замороженный выбор → validation

Пул определяется **raw fusion MVP/R1 50:50**, а не отдельным R1-кандидатом. Внутри пула сохраняется относительный порядок полного графа v25 (k1=20, k2=3, λ=0.50); снаружи остаётся raw-порядок. Граф не перестраивается на усечённой gallery.

При равенстве mAP сохраняется контроль. Если он победит, повторный validation того же варианта не нужен. Порог, confidence и отказы неизменны. Показываются этап, номер сравнения, mAP, время и использование готовых результатов. Лимита времени нет.

После прерывания снова выполни Restart Kernel → Run All с тем же RUN_NAME. Повреждённый кэш не заменяется молча; ошибки сохраняются в tasks/.../error.json.

In [4]:
results = experiment.run(context, allow_outer=ALLOW_OUTER_EVALUATION)
assert results['status'] == 'complete' and results['protected_unchanged']
assert not results['promoted'] and not results['threshold_fit']
print('Завершено. Выбор по calibration:', results['selection']['selected']['name'])
print('Время текущего вызова:', round(results['elapsed_seconds'] / 60, 2), 'мин')


CALIBRATION 1/4 | elapsed 10.8s
[START] calibration_V25_control | elapsed 0.00 h
calibration V25_control: mAP=0.826410; R1 candidates unchanged
[DONE] calibration_V25_control | 0.0 min

CALIBRATION 2/4 | elapsed 11.1s
[START] calibration_raw_pool10 | elapsed 0.00 h
calibration raw_pool10: mAP=0.815951; R1 candidates unchanged
[DONE] calibration_raw_pool10 | 0.0 min

CALIBRATION 3/4 | elapsed 11.4s
[START] calibration_raw_pool20 | elapsed 0.00 h
calibration raw_pool20: mAP=0.824781; R1 candidates unchanged
[DONE] calibration_raw_pool20 | 0.0 min

CALIBRATION 4/4 | elapsed 11.7s
[START] calibration_raw_pool50 | elapsed 0.00 h
calibration raw_pool50: mAP=0.826995; R1 candidates unchanged
[DONE] calibration_raw_pool50 | 0.0 min

FROZEN: raw_pool50; validation v25 + winner only

VALIDATION 1/2 | elapsed 12.1s
[START] validation_V25_control | elapsed 0.00 h
validation V25_control: mAP=0.828957; R1 candidates unchanged
[DONE] validation_V25_control | 0.0 min

VALIDATION 2/2 | elapsed 13.0s
[

## 4. Отчёт и артефакты

Итоговые мAP/Rank-1 и изменение AP по запросам — ниже. Для контроля и выбранного варианта сохраняются submission.csv, candidates.csv и реальные embeddings.npy. Даже при отказе ranking содержит десять ID. Ограничение junk/top-10 остаётся прежним; evaluator и bbox не меняются.

После завершения пришли сообщение «подсчёты завершились». Решение о переносе в MVP принимаем отдельно по полученным результатам.

In [5]:
display(Markdown((context['output'] / 'REPORT.md').read_text()))
display(results['selected_vs_v25'])
for name, report in results['evaluations'].items():
    print(name, '→', report['export'])
print('Основные файлы:', context['output'] / 'results.json', context['output'] / 'REPORT.md')

# v30 — ограничение реранкинга исходным пулом

Статус: complete
Четыре варианта: v25, raw top-10/20/50; fusion 50:50, граф 20/3/λ0.50 неизменны.
Пул выбирается по raw cosine fusion, не по отдельному raw R1-кандидату.
Внутри пула — порядок полного графа v25; за его пределами — исходный raw-порядок.
Все четыре сравнения на calibration; на validation — контроль и один замороженный выбор.

## Calibration

| Вариант | mAP@10 |
|---|---:|
| V25_control | 0.826410 |
| raw_pool10 | 0.815951 |
| raw_pool20 | 0.824781 |
| raw_pool50 | 0.826995 |

Выбор: raw_pool50

## Validation

| Вариант | mAP@10 | Rank-1 | F1 | TNR |
|---|---:|---:|---:|---:|
| V25_control | 0.828957 | 0.829960 | 0.795455 | 0.709677 |
| raw_pool50 | 0.828340 | 0.829960 | 0.795455 | 0.709677 |

ΔmAP против v25: -0.000617. AP лучше/хуже/равно: 1/2/244.

## Ограничения

Адаптивный поиск после изучения ошибок на development-данных, не независимый тест и не обещание hidden-test прироста.
Веса, эмбеддинги, кандидаты, confidence, порог, bbox, evaluator и рабочий MVP не изменяются.
Обучения, нового извлечения признаков и автоматического продвижения нет. Каждый query независим.
Экспорт: десять уникальных gallery-ID даже при отказе, отказ — отсутствие строки кандидата, реальные float32-векторы.
Ограничение junk/top-10 остаётся: evaluator фильтрует junk, но 11-е место из файла top-10 не восстановить.
Источники неизменны: True. Время текущего вызова: 0.2 мин.


{'mAP_delta': -0.0006169269327164217,
 'C_delta': 0.0,
 'improved': 1,
 'worsened': 2,
 'unchanged': 244}

V25_control → /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_30_pool_rerank/runs/pool_rerank_v1/tasks/validation_V25_control/export
raw_pool50 → /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_30_pool_rerank/runs/pool_rerank_v1/tasks/validation_raw_pool50/export
Основные файлы: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_30_pool_rerank/runs/pool_rerank_v1/results.json /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_30_pool_rerank/runs/pool_rerank_v1/REPORT.md
